# scimap — LongMemEval Benchmark

Reproduces the **94.7% recall_all@10** result on [LongMemEval-S](https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned).

**What this measures:** Session-level retrieval recall — given a question and 53 conversation sessions (1-4 containing the answer, the rest filler), does scimap retrieve chunks from the correct sessions?

**Runtime:** ~30 minutes for 500 questions, ~4 minutes for 50 questions.

Paper: [LongMemEval (ICLR 2025)](https://arxiv.org/abs/2410.10813)

In [ ]:
# Step 1: Clone scimap
!git clone https://github.com/kunal12203/swafra /content/scimap
%cd /content/scimap/packages/mcp

In [ ]:
# Step 2: Install dependencies (no GPU needed)
!pip install -q -r engine/requirements.txt

In [ ]:
# Step 3: Download LongMemEval-S dataset from HuggingFace
import os
os.makedirs('bench/data', exist_ok=True)
!curl -L -o bench/data/longmemeval_s_cleaned.json \
  https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json
print('Dataset downloaded.')

In [ ]:
# Step 4: Run the benchmark
# Args: <num_questions> <k>
# Use 50 for a quick run (~4 min), 500 for full benchmark (~30 min)
import subprocess
result = subprocess.run(
    ['python', 'bench/run_eval.py', '50', '10'],
    capture_output=True, text=True,
    env={**os.environ, 'SCIMAP_EMBED_BACKEND': 'local', 'SCIMAP_DATA_DIR': '/tmp/scimap-bench'}
)
# Print only the results (filter out warnings)
for line in result.stdout.splitlines():
    print(line)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Step 5 (optional): Run full 500-question benchmark
# Uncomment to run — takes ~30 minutes
# !python bench/run_eval.py 500 10

In [ ]:
# Step 6: View detailed results
import json
with open('bench/results.json') as f:
    results = json.load(f)

print('=== SUMMARY ===')
s = results['summary']
print(f"Evaluated: {s['evaluated']} questions")
print(f"recall_any@10:  {s['recall_any']*100:.1f}%")
print(f"recall_all@10:  {s['recall_all']*100:.1f}%")
print(f"recall_frac@10: {s['recall_fraction']*100:.1f}%")
print()
print('=== BY CATEGORY ===')
for qtype, m in sorted(results['by_type'].items()):
    print(f"  {qtype:30s}: any={m['recall_any']*100:.1f}%  all={m['recall_all']*100:.1f}%  (n={m['count']})")